# Drive mounting

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/P9/Src')

# Importing packages

In [ ]:
import os
import glob
import json
import random
import zipfile

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, concatenate, 
    Conv2DTranspose, BatchNormalization, Activation
)
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
)

In [ ]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"✅ Memory Growth activée sur {len(gpus)} GPU(s)")
    except RuntimeError as e:
        print(e)

# Chargement des données

In [ ]:
isic_dir_drive = "/content/drive/MyDrive/P9/Src/Data/ISIC2018"
local_base_dir = "/content/P9/Data/ISIC2018"

zip_files = glob.glob(os.path.join(isic_dir_drive, "**/*.zip"), recursive=True)

if not zip_files:
    print(f"❌ Aucun fichier ZIP trouvé dans {isic_dir_drive}")
else:
    print(f"📦 {len(zip_files)} fichiers ZIP trouvés. Décompression...")
    for zip_path in zip_files:
        sub_folder = os.path.relpath(os.path.dirname(zip_path), isic_dir_drive)
        local_extract_dir = os.path.join(local_base_dir, sub_folder)
        os.makedirs(local_extract_dir, exist_ok=True)
        print(f"Extraction de {os.path.basename(zip_path)} dans {local_extract_dir}...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(local_extract_dir)
    print(f"✅ Extraction terminée dans {local_base_dir}")

# Definition des paths

In [ ]:
train_images_dir = '/content/P9/Data/ISIC2018/train/Training_Input'
train_masks_dir = '/content/P9/Data/ISIC2018/train/Training_GroundTruth'
validation_images_dir = '/content/P9/Data/ISIC2018/validation/Validation_Input'
validation_masks_dir = '/content/P9/Data/ISIC2018/validation/Validation_GroundTruth'
test_images_dir = '/content/P9/Data/ISIC2018/test/Test_Input'
test_masks_dir = '/content/P9/Data/ISIC2018/test/Test_GroundTruth'

train_images_paths = sorted(glob.glob(os.path.join(train_images_dir, '*.jpg')))
train_masks_paths = sorted(glob.glob(os.path.join(train_masks_dir, '*.png')))
validation_images_paths = sorted(glob.glob(os.path.join(validation_images_dir, '*.jpg')))
validation_masks_paths = sorted(glob.glob(os.path.join(validation_masks_dir, '*.png')))
test_images_paths = sorted(glob.glob(os.path.join(test_images_dir, '*.jpg')))
test_masks_paths = sorted(glob.glob(os.path.join(test_masks_dir, '*.png')))

print(f"Train: {len(train_images_paths)} images, {len(train_masks_paths)} masques")
print(f"Validation: {len(validation_images_paths)} images, {len(validation_masks_paths)} masques")
print(f"Test: {len(test_images_paths)} images, {len(test_masks_paths)} masques")
print("Vérification :", os.path.basename(train_images_paths[0]), "->", os.path.basename(train_masks_paths[0]))

# Data loaders

In [ ]:
IMG_HEIGHT = 256
IMG_WIDTH = 256
BATCH_SIZE = 32

class ISIC2018Generator(tf.keras.utils.Sequence):
    def __init__(self, img_paths, mask_paths, batch_size=BATCH_SIZE, img_size=(IMG_HEIGHT, IMG_WIDTH)):
        self.img_paths = list(img_paths)
        self.mask_paths = list(mask_paths)
        self.batch_size = batch_size
        self.img_size = img_size

    def __len__(self):
        return int(np.ceil(len(self.img_paths) / self.batch_size))

    def __getitem__(self, idx):
        batch_img = self.img_paths[idx * self.batch_size : (idx + 1) * self.batch_size]
        batch_mask = self.mask_paths[idx * self.batch_size : (idx + 1) * self.batch_size]
        
        x = np.zeros((len(batch_img), self.img_size[0], self.img_size[1], 3), dtype="float32")
        y = np.zeros((len(batch_img), self.img_size[0], self.img_size[1], 1), dtype="float32")
        
        for j, (img_path, mask_path) in enumerate(zip(batch_img, batch_mask)):
            img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
            x[j] = cv2.resize(img, self.img_size) / 255.0
            
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            mask = cv2.resize(mask, self.img_size, interpolation=cv2.INTER_NEAREST)
            y[j] = np.expand_dims(np.where(mask > 127.5, 1.0, 0.0), -1)
            
        return x, y

    def on_epoch_end(self):
        combined = list(zip(self.img_paths, self.mask_paths))
        random.shuffle(combined)
        shuffled = list(zip(*combined))
        if shuffled:
            self.img_paths, self.mask_paths = list(shuffled[0]), list(shuffled[1])

train_dataset = ISIC2018Generator(train_images_paths, train_masks_paths, batch_size=BATCH_SIZE)
val_dataset = ISIC2018Generator(validation_images_paths, validation_masks_paths, batch_size=BATCH_SIZE)
test_dataset = ISIC2018Generator(test_images_paths, test_masks_paths, batch_size=BATCH_SIZE)
print(f"✅ DataGenerators prêts ! Batch size : {BATCH_SIZE}")

In [ ]:
images, masks = train_dataset[0]
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.title("Image redimensionnée")
plt.imshow(images[0])
plt.axis('off')

plt.subplot(1, 2, 2)
plt.title("Masque binarisé")
plt.imshow(masks[0].squeeze(), cmap='gray')
plt.axis('off')

plt.show()

# Unet model

In [ ]:
def conv_block(input_tensor, num_filters):
    x = Conv2D(num_filters, (3, 3), padding='same')(input_tensor)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Conv2D(num_filters, (3, 3), padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    return x

def build_unet(input_shape=(256, 256, 3)):
    inputs = Input(input_shape)
    
    conv1 = conv_block(inputs, 64)
    pool1 = MaxPooling2D(pool_size=(2, 2))(conv1)
    
    conv2 = conv_block(pool1, 128)
    pool2 = MaxPooling2D(pool_size=(2, 2))(conv2)
    
    conv3 = conv_block(pool2, 256)
    pool3 = MaxPooling2D(pool_size=(2, 2))(conv3)
    
    conv4 = conv_block(pool3, 512)
    pool4 = MaxPooling2D(pool_size=(2, 2))(conv4)
    
    conv5 = conv_block(pool4, 1024)
    
    up6 = Conv2DTranspose(512, (2, 2), strides=(2, 2), padding='same')(conv5)
    up6 = concatenate([up6, conv4])
    conv6 = conv_block(up6, 512)
    
    up7 = Conv2DTranspose(256, (2, 2), strides=(2, 2), padding='same')(conv6)
    up7 = concatenate([up7, conv3])
    conv7 = conv_block(up7, 256)
    
    up8 = Conv2DTranspose(128, (2, 2), strides=(2, 2), padding='same')(conv7)
    up8 = concatenate([up8, conv2])
    conv8 = conv_block(up8, 128)
    
    up9 = Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same')(conv8)
    up9 = concatenate([up9, conv1])
    conv9 = conv_block(up9, 64)
    
    outputs = Conv2D(1, (1, 1), activation='sigmoid')(conv9)
    return Model(inputs=[inputs], outputs=[outputs])

unet_model = build_unet(input_shape=(IMG_HEIGHT, IMG_WIDTH, 3))
unet_model.summary()

# Metriques

In [ ]:
def dice_coef(y_true, y_pred, smooth=1e-6):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

def dice_loss(y_true, y_pred):
    return 1 - dice_coef(y_true, y_pred)

def iou(y_true, y_pred, smooth=1e-6):
    intersection = K.sum(K.abs(y_true * y_pred), axis=[1,2,3])
    union = K.sum(y_true,[1,2,3]) + K.sum(y_pred,[1,2,3]) - intersection
    return K.mean((intersection + smooth) / (union + smooth), axis=0)

In [ ]:
unet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=dice_loss, 
    metrics=[dice_coef, iou, 'accuracy']
)

# Callbacks et entrainement

In [ ]:
checkpoint_filepath = '/content/drive/MyDrive/P9/Src/models/best_unet_isic2018.h5'

callbacks = [
    ModelCheckpoint(checkpoint_filepath, monitor='val_loss', save_best_only=True, mode='min', verbose=1),
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1)
]

print("Début de l'entraînement de U-Net...")
history = unet_model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=100,
    callbacks=callbacks
)

Courbes d'apprentissage

In [ ]:
def plot_training_history(history):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    ax1.plot(history.history['loss'], label='Entraînement (Loss)')
    ax1.plot(history.history['val_loss'], label='Validation (Loss)')
    ax1.set_title('Évolution de la Dice Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()
    
    ax2.plot(history.history['iou'], label='Entraînement (IoU)')
    ax2.plot(history.history['val_iou'], label='Validation (IoU)')
    ax2.set_title('Évolution de l'IoU')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('IoU Score')
    ax2.legend()
    
    plt.show()

plot_training_history(history)

# Visualisation des prédictions

In [ ]:
def visualize_predictions(dataset, model, num_samples=3):
    images, masks = dataset[0]
    predictions = model.predict(images[:num_samples])
    preds_bin = (predictions > 0.5).astype(np.float32)
    
    plt.figure(figsize=(15, 5 * num_samples))
    for i in range(num_samples):
        plt.subplot(num_samples, 3, i*3 + 1)
        plt.title("Image Originale (Test)")
        plt.imshow(images[i])
        plt.axis('off')
        
        plt.subplot(num_samples, 3, i*3 + 2)
        plt.title("Vérité Terrain (Masque Réel)")
        plt.imshow(masks[i].squeeze(), cmap='gray')
        plt.axis('off')
        
        plt.subplot(num_samples, 3, i*3 + 3)
        plt.title("Prédiction U-Net")
        plt.imshow(preds_bin[i].squeeze(), cmap='gray')
        plt.axis('off')
        
    plt.tight_layout()
    plt.show()

visualize_predictions(test_dataset, unet_model, num_samples=4)

# Traçabilité des metriques et test

In [ ]:
exp_dir = '/content/drive/MyDrive/P9/Src/Experiences/Unet'
os.makedirs(exp_dir, exist_ok=True)

history_df = pd.DataFrame(history.history)
history_csv_path = os.path.join(exp_dir, 'unet_training_history.csv')
history_df.to_csv(history_csv_path, index=False)
print(f"✅ Historique d'entraînement sauvegardé dans : {history_csv_path}")

print("\nÉvaluation sur l'ensemble de Test...")
unet_model.load_weights(checkpoint_filepath) 
test_results = unet_model.evaluate(test_dataset, verbose=1)
metrics_names = unet_model.metrics_names 

test_scores_path = os.path.join(exp_dir, 'unet_test_scores.txt')
with open(test_scores_path, 'w') as f:
    f.write("=== RÉSULTATS DE U-NET SUR ISIC 2018 (TEST SET) ===\n")
    for name, score in zip(metrics_names, test_results):
        f.write(f"{name}: {score:.4f}\n")
        print(f"{name}: {score:.4f}")
print(f"✅ Scores de test sauvegardés dans : {test_scores_path}")

images, masks = test_dataset[0]
predictions = unet_model.predict(images[:2])
preds_bin = (predictions > 0.5).astype(np.float32)

plt.figure(figsize=(15, 10))
for i in range(2):
    plt.subplot(2, 3, i*3 + 1)
    plt.title("Image Originale")
    plt.imshow(images[i])
    plt.axis('off')
    
    plt.subplot(2, 3, i*3 + 2)
    plt.title("Vérité Terrain")
    plt.imshow(masks[i].squeeze(), cmap='gray')
    plt.axis('off')
    
    plt.subplot(2, 3, i*3 + 3)
    plt.title("Prédiction U-Net")
    plt.imshow(preds_bin[i].squeeze(), cmap='gray')
    plt.axis('off')

plt.tight_layout()
predictions_img_path = os.path.join(exp_dir, 'unet_test_predictions.png')
plt.savefig(predictions_img_path)
print(f"✅ Image de prédictions sauvegardée dans : {predictions_img_path}")
plt.show()